In [2]:
!pip install dagshub mlflow optuna -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 100.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from kaggle_secrets import UserSecretsClient
import warnings
warnings.filterwarnings('ignore')

dagshub.init (
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

mlflow.set_experiment("Neural Networks")
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=ab9e1940-daca-4b09-a93c-3685a8b21b18&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e8f5009c60bb5507fce9bb3f654481bf23d1fee4bfde39c48fa88f2e060f577f




Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow


# Cleaning

In [3]:
with mlflow.start_run(run_name="NeuralNetworks_Cleaning"):
    import gc
    import numpy as np
    import pandas as pd

    DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
    TXN_MISSING_THRESHOLD = 0.80
    ID_MISSING_THRESHOLD = 0.95
    NEAR_CONSTANT_THRESHOLD = 0.99

    # load
    train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
    train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
    test_txn  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
    test_id   = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

    # fix id-01 vs id_01
    test_id.columns = test_id.columns.str.replace("-", "_", regex=False)

    # merge
    train = train_txn.merge(train_id, on="TransactionID", how="left")
    test  = test_txn.merge(test_id, on="TransactionID", how="left")

    del train_txn, train_id, test_txn, test_id
    gc.collect()

    # split target
    y_train = train["isFraud"].astype(np.int8)
    X_train = train.drop(columns=["isFraud", "TransactionID"])
    X_test  = test.drop(columns=["TransactionID"])

    del train, test
    gc.collect()

    # drop high-missing
    id_like_cols = [c for c in X_train.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_like_cols = [c for c in X_train.columns if c not in id_like_cols]
    missing_ratio = X_train.isnull().mean()

    drop_txn = [c for c in txn_like_cols if missing_ratio[c] > TXN_MISSING_THRESHOLD]
    drop_id  = [c for c in id_like_cols if missing_ratio[c] > ID_MISSING_THRESHOLD]
    drop_missing = drop_txn + drop_id

    X_train = X_train.drop(columns=drop_missing)
    X_test  = X_test.drop(columns=[c for c in drop_missing if c in X_test.columns])

    # drop near-constant
    near_constant_cols = []
    for col in X_train.columns:
        top_freq = X_train[col].value_counts(dropna=False, normalize=True).iloc[0]
        if top_freq > NEAR_CONSTANT_THRESHOLD:
            near_constant_cols.append(col)

    X_train = X_train.drop(columns=near_constant_cols)
    X_test  = X_test.drop(columns=[c for c in near_constant_cols if c in X_test.columns])

    # align test columns
    for col in X_train.columns:
        if col not in X_test.columns:
            X_test[col] = np.nan
    X_test = X_test[X_train.columns]

    # log
    mlflow.log_param("txn_missing_threshold", TXN_MISSING_THRESHOLD)
    mlflow.log_param("id_missing_threshold", ID_MISSING_THRESHOLD)
    mlflow.log_param("near_constant_threshold", NEAR_CONSTANT_THRESHOLD)

    mlflow.log_metric("train_rows", int(X_train.shape[0]))
    mlflow.log_metric("test_rows", int(X_test.shape[0]))
    mlflow.log_metric("final_features", int(X_train.shape[1]))
    mlflow.log_metric("fraud_rate", float(y_train.mean()))
    mlflow.log_metric("dropped_missing_cols", int(len(drop_missing)))
    mlflow.log_metric("dropped_near_constant_cols", int(len(near_constant_cols)))

    print(f"X_train_clean: {X_train.shape}")
    print(f"X_test_clean:  {X_test.shape}")

    # keep for next cells
    X_train_clean = X_train
    X_test_clean = X_test
    y_train_clean = y_train

X_train_clean: (590540, 353)
X_test_clean:  (506691, 353)
🏃 View run NeuralNetworks_Cleaning at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/4bd87bfd9185417f86cd34210d46eeaa
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5


# Feature Engineering

In [4]:
with mlflow.start_run(run_name="NeuralNetworks_FeatureEngineering"):
    from sklearn.impute import SimpleImputer

    X_train = X_train_clean.copy()
    X_test = X_test_clean.copy()
    y_train = y_train_clean.copy()

    X_train["TransactionAmt_log"] = np.log1p(X_train["TransactionAmt"].clip(lower=0))
    X_test["TransactionAmt_log"] = np.log1p(X_test["TransactionAmt"].clip(lower=0))

    X_train["hour_sin"] = np.sin(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_train["hour_cos"] = np.cos(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_sin"] = np.sin(2 * np.pi * ((X_test["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_cos"] = np.cos(2 * np.pi * ((X_test["TransactionDT"] // 3600) % 24) / 24)

    X_train = X_train.drop(columns=["TransactionDT"], errors="ignore")
    X_test = X_test.drop(columns=["TransactionDT"], errors="ignore")

    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    num_imp = SimpleImputer(strategy="median")
    X_train[num_cols] = num_imp.fit_transform(X_train[num_cols])
    X_test[num_cols] = num_imp.transform(X_test[num_cols])

    cat_maps = {}
    for c in cat_cols:
        uniq = pd.Series(X_train[c].astype(str).unique())
        mapping = {v: i for i, v in enumerate(uniq)}
        cat_maps[c] = mapping
        X_train[c] = X_train[c].astype(str).map(mapping).fillna(-1).astype(np.int32)
        X_test[c] = X_test[c].astype(str).map(mapping).fillna(-1).astype(np.int32)

    X_test = X_test.reindex(columns=X_train.columns, fill_value=-1)

    mlflow.log_metric("features_after_fe", int(X_train.shape[1]))
    mlflow.log_metric("cat_cols_encoded", int(len(cat_cols)))

    print("X_train_fe:", X_train.shape)
    print("X_test_fe: ", X_test.shape)

    X_train_fe_nn = X_train
    X_test_fe_nn = X_test
    y_train_fe_nn = y_train

X_train_fe: (590540, 355)
X_test_fe:  (506691, 355)
🏃 View run NeuralNetworks_FeatureEngineering at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/185f9122b35645c5878712d7b8f6407f
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5


In [5]:
print(X_train_fe_nn.shape, X_test_fe_nn.shape)
assert X_train_fe_nn.shape[1] == X_test_fe_nn.shape[1]
print("object cols left:", X_train_fe_nn.select_dtypes(include=["object"]).shape[1])

(590540, 355) (506691, 355)
object cols left: 0


# Feature Selection

In [6]:
with mlflow.start_run(run_name="NeuralNetworks_FeatureSelection"):
    X_train = X_train_fe_nn.copy()
    X_test = X_test_fe_nn.copy()

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    nu = X_train.nunique(dropna=False)
    const_cols = nu[nu <= 1].index.tolist()
    X_train = X_train.drop(columns=const_cols, errors="ignore")
    X_test = X_test.drop(columns=const_cols, errors="ignore")

    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    sample_n = min(120_000, len(X_train))
    idx = np.random.RandomState(42).choice(len(X_train), size=sample_n, replace=False)
    corr = X_train.iloc[idx][num_cols].corr().abs()

    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop_corr = [c for c in upper.columns if (upper[c] > 0.98).any()]

    X_train = X_train.drop(columns=drop_corr, errors="ignore")
    X_test = X_test.drop(columns=drop_corr, errors="ignore")

    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    mlflow.log_metric("dropped_const", len(const_cols))
    mlflow.log_metric("dropped_corr", len(drop_corr))
    mlflow.log_metric("features_after_fs", int(X_train.shape[1]))

    print("X_train_fs:", X_train.shape)

    X_train_final_nn = X_train
    X_test_final_nn = X_test

X_train_fs: (590540, 306)
🏃 View run NeuralNetworks_FeatureSelection at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/2015c707850645dfac73561f8628138b
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5


In [7]:
print(X_train_final_nn.shape, X_test_final_nn.shape)
assert X_train_final_nn.shape[1] == X_test_final_nn.shape[1]

(590540, 306) (506691, 306)


In [10]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final_nn, y_train_fe_nn,
    test_size=0.2, random_state=42, stratify=y_train_fe_nn
)
print(f"Train: {X_tr.shape}, Val: {X_val.shape}")
print(f"Fraud rate train: {y_tr.mean():.4f}, val: {y_val.mean():.4f}")

import pickle, os
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

with open(f"{CHECKPOINT_DIR}/X_train_final_nn.pkl", "wb") as f:
    pickle.dump(X_train_final_nn, f)
with open(f"{CHECKPOINT_DIR}/X_test_final_nn.pkl", "wb") as f:
    pickle.dump(X_test_final_nn, f)
with open(f"{CHECKPOINT_DIR}/y_train_fe_nn.pkl", "wb") as f:
    pickle.dump(y_train_fe_nn, f)

print("✅ Checkpoint saved to", CHECKPOINT_DIR)

Train: (472432, 306), Val: (118108, 306)
Fraud rate train: 0.0350, val: 0.0350
✅ Checkpoint saved to /kaggle/working/checkpoints


In [4]:
import pickle, os

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
CHECKPOINT_EXISTS = all(
    os.path.exists(f"{CHECKPOINT_DIR}/{f}")
    for f in ["X_train_final_nn.pkl", "X_test_final_nn.pkl", "y_train_fe_nn.pkl"]
)

if CHECKPOINT_EXISTS:
    with open(f"{CHECKPOINT_DIR}/X_train_final_nn.pkl", "rb") as f:
        X_train_final_nn = pickle.load(f)
    with open(f"{CHECKPOINT_DIR}/X_test_final_nn.pkl", "rb") as f:
        X_test_final_nn = pickle.load(f)
    with open(f"{CHECKPOINT_DIR}/y_train_fe_nn.pkl", "rb") as f:
        y_train_fe_nn = pickle.load(f)
    print("✅ Loaded from checkpoint — skipped preprocessing")
else:
    print("⚠️  No checkpoint found — run preprocessing cells first")

⚠️  No checkpoint found — run preprocessing cells first


# **Training**

In [12]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
import pickle

SAMPLE_SIZE = 150_000
sample_idx = np.random.RandomState(42).choice(len(X_train_final_nn), size=SAMPLE_SIZE, replace=False)
X_sample = X_train_final_nn.iloc[sample_idx].reset_index(drop=True)
y_sample = y_train_fe_nn.iloc[sample_idx].reset_index(drop=True)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_sample, y_sample,
    test_size=0.2, random_state=42, stratify=y_sample
)
print(f"Train: {X_tr.shape} | Val: {X_val.shape}")
print(f"Fraud rate train: {y_tr.mean():.4f} | val: {y_val.mean():.4f}")

Train: (120000, 306) | Val: (30000, 306)
Fraud rate train: 0.0359 | val: 0.0359


In [13]:
with mlflow.start_run(run_name="MLP_Baseline"):
    mlflow.log_param("hidden_layer_sizes", "(100,)")
    mlflow.log_param("activation",         "relu")
    mlflow.log_param("solver",             "adam")
    mlflow.log_param("max_iter",           100)
    mlflow.log_param("sample_size",        SAMPLE_SIZE)
    mlflow.log_param("note",               "single hidden layer, default params")

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    MLPClassifier(hidden_layer_sizes=(100,), activation="relu",
                                  solver="adam", max_iter=100, random_state=42))
    ])
    pipe.fit(X_tr, y_tr)

    train_auc = roc_auc_score(y_tr,  pipe.predict_proba(X_tr)[:, 1])
    val_auc   = roc_auc_score(y_val, pipe.predict_proba(X_val)[:, 1])

    mlflow.log_metric("train_auc",   round(train_auc, 5))
    mlflow.log_metric("val_auc",     round(val_auc,   5))
    mlflow.log_metric("overfit_gap", round(train_auc - val_auc, 5))

    print(f"[Baseline] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc - val_auc:.4f}")

[Baseline] Train: 0.9943 | Val: 0.8271 | Gap: 0.1672
🏃 View run MLP_Baseline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/75582ba5f4d043328bdda12e43f57cec
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5


In [14]:
architectures = [
    (100,),
    (128, 64),
    (256, 128),
]

arch_results = []
for arch in architectures:
    label = "x".join(str(x) for x in arch)
    with mlflow.start_run(run_name=f"MLP_arch_{label}"):
        mlflow.log_param("hidden_layer_sizes", label)
        mlflow.log_param("activation",         "relu")
        mlflow.log_param("solver",             "adam")
        mlflow.log_param("max_iter",           150)
        mlflow.log_param("sample_size",        SAMPLE_SIZE)

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    MLPClassifier(hidden_layer_sizes=arch, activation="relu",
                                      solver="adam", max_iter=150, random_state=42))
        ])
        pipe.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  pipe.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, pipe.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        arch_results.append({"arch": label, "train_auc": train_auc,
                              "val_auc": val_auc, "gap": gap})
        print(f"  {label:<15} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

arch_df = pd.DataFrame(arch_results).set_index("arch")
print(f"\n-> Best arch by val AUC: {arch_df['val_auc'].idxmax()}")
print(arch_df.to_string())

  100             | Train: 0.9961 | Val: 0.8107 | Gap: 0.1853
🏃 View run MLP_arch_100 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/4ada0fc80d374237bba252e1a7c63de7
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5
  128x64          | Train: 0.9993 | Val: 0.8383 | Gap: 0.1610
🏃 View run MLP_arch_128x64 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/93d50feaa4d64b0e9608b0c1c1691aa0
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5
  256x128         | Train: 0.9997 | Val: 0.8523 | Gap: 0.1475
🏃 View run MLP_arch_256x128 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/0f61b0ad86d94c6db37c1677de294287
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5

-> Best arch by val AUC: 256x128
         train_auc   val_auc       gap
ar

In [15]:
best_arch = (256, 128)
best_activation = "relu"
print(f"Using arch: {best_arch}, activation: {best_activation}")

Using arch: (256, 128), activation: relu


In [16]:
alpha_results = []
for alpha in [0.0001, 0.001, 0.01, 0.1]:
    with mlflow.start_run(run_name=f"MLP_alpha_{alpha}"):
        mlflow.log_param("hidden_layer_sizes", str(best_arch))
        mlflow.log_param("activation",         best_activation)
        mlflow.log_param("alpha",              alpha)
        mlflow.log_param("solver",             "adam")
        mlflow.log_param("max_iter",           150)
        mlflow.log_param("sample_size",        SAMPLE_SIZE)

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    MLPClassifier(hidden_layer_sizes=best_arch,
                                      activation=best_activation,
                                      alpha=alpha, solver="adam",
                                      max_iter=150, random_state=42))
        ])
        pipe.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  pipe.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, pipe.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        alpha_results.append({"alpha": alpha, "train_auc": train_auc,
                               "val_auc": val_auc, "gap": gap})
        print(f"  alpha={alpha:<6} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

alpha_df = pd.DataFrame(alpha_results).set_index("alpha")
print(f"\n-> Best alpha: {alpha_df['val_auc'].idxmax()}")
print(alpha_df.to_string())

  alpha=0.0001 | Train: 0.9997 | Val: 0.8523 | Gap: 0.1475
🏃 View run MLP_alpha_0.0001 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/3034cf77c86d4cc4b9e5b8822aed0ea8
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5
  alpha=0.001  | Train: 0.9995 | Val: 0.8561 | Gap: 0.1434
🏃 View run MLP_alpha_0.001 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/c263c68971494cf0a6caf54cbe29bb1b
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5
  alpha=0.01   | Train: 0.9984 | Val: 0.8669 | Gap: 0.1315
🏃 View run MLP_alpha_0.01 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/2f071cca906444159aa129c55b353a71
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5
  alpha=0.1    | Train: 0.9585 | Val: 0.8787 | Gap: 0.0797
🏃 View run MLP_alpha_0.

In [17]:
import pickle
import os

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

mlp_baseline_data = {
    "run_name": "MLP_Baseline",
    "train_auc": train_auc, 
    "val_auc": val_auc,
    "gap": train_auc - val_auc
}
with open(f"{CHECKPOINT_DIR}/mlp_baseline.pkl", "wb") as f:
    pickle.dump(mlp_baseline_data, f)

with open(f"{CHECKPOINT_DIR}/mlp_alpha_results.pkl", "wb") as f:
    pickle.dump(alpha_df, f)

with open(f"{CHECKPOINT_DIR}/mlp_arch_results.pkl", "wb") as f:
    pickle.dump(arch_df, f)

print(f"✅ Baseline saved to: {CHECKPOINT_DIR}/mlp_baseline.pkl")
print(f"✅ Alpha results saved to: {CHECKPOINT_DIR}/mlp_alpha_results.pkl")
print(f"✅ Architecture results saved to: {CHECKPOINT_DIR}/mlp_arch_results.pkl")

✅ Baseline saved to: /kaggle/working/checkpoints/mlp_baseline.pkl
✅ Alpha results saved to: /kaggle/working/checkpoints/mlp_alpha_results.pkl
✅ Architecture results saved to: /kaggle/working/checkpoints/mlp_arch_results.pkl


In [18]:
best_arch = (256, 128)
best_activation = "relu"
best_alpha = 0.1
print(f"arch: {best_arch}, activation: {best_activation}, alpha: {best_alpha}")

arch: (256, 128), activation: relu, alpha: 0.1


In [6]:
import pickle
import os
import pandas as pd

CHECKPOINT_DIR = "/kaggle/working/checkpoints"

files = {
    "baseline": "mlp_baseline.pkl",
    "alpha": "mlp_alpha_results.pkl",
    "arch": "mlp_arch_results.pkl"
}

if all(os.path.exists(f"{CHECKPOINT_DIR}/{name}") for name in files.values()):
    
    with open(f"{CHECKPOINT_DIR}/{files['baseline']}", "rb") as f:
        baseline_data = pickle.load(f)
        train_auc_base = baseline_data['train_auc']
        val_auc_base = baseline_data['val_auc']
    
    with open(f"{CHECKPOINT_DIR}/{files['alpha']}", "rb") as f:
        alpha_df = pickle.load(f)
        
    with open(f"{CHECKPOINT_DIR}/{files['arch']}", "rb") as f:
        arch_df = pickle.load(f)

    print("✅ All MLP checkpoints loaded successfully!")
    print(f"Baseline Val AUC was: {val_auc_base:.4f}")
else:
    print("⚠️ Checkpoint files missing. You need to run the training loops first.")

⚠️ Checkpoint files missing. You need to run the training loops first.


In [7]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
import mlflow

sample_idx = np.random.choice(len(X_train), size=150_000, replace=False)
X_sample = X_train.iloc[sample_idx].values
y_sample = y_train.iloc[sample_idx].values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []

with mlflow.start_run(run_name="MLP_CrossValidation_5fold"):
    mlflow.log_param("hidden_layer_sizes", str(best_arch))
    mlflow.log_param("activation",         best_activation)
    mlflow.log_param("alpha",              best_alpha)
    mlflow.log_param("solver",             "adam")
    mlflow.log_param("max_iter",           150)
    mlflow.log_param("cv_folds",           5)
    mlflow.log_param("cv_sample_size",     150_000)

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_sample, y_sample)):
        X_fold_train, X_fold_val = X_sample[train_idx], X_sample[val_idx]
        y_fold_train, y_fold_val = y_sample[train_idx], y_sample[val_idx]

        model = MLPClassifier(
            hidden_layer_sizes=best_arch,
            activation=best_activation,
            alpha=best_alpha,
            solver="adam",
            max_iter=150,
            random_state=42
        )
        model.fit(X_fold_train, y_fold_train)
        fold_auc = roc_auc_score(y_fold_val, model.predict_proba(X_fold_val)[:, 1])
        fold_scores.append(fold_auc)
        mlflow.log_metric("fold_auc", fold_auc, step=fold)
        print(f"Fold {fold+1} AUC: {fold_auc:.5f}")

    mlflow.log_metric("cv_mean_auc", np.mean(fold_scores))
    mlflow.log_metric("cv_std_auc",  np.std(fold_scores))
    print(f"\nCV Mean AUC: {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}")

NameError: name 'X_train' is not defined

In [1]:
from sklearn.neural_network import MLPClassifier

with mlflow.start_run(run_name="MLP_Final_Pipeline") as run:
    mlflow.log_param("hidden_layer_sizes", str(best_arch))
    mlflow.log_param("activation",         best_activation)
    mlflow.log_param("alpha",              best_alpha)
    mlflow.log_param("solver",             "adam")
    mlflow.log_param("max_iter",           150)
    mlflow.log_param("trained_on",         "120k_stratified_sample")
    mlflow.log_param("note",               "full 590k not feasible on CPU for MLP — trained on 150k sample")

    final_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    MLPClassifier(hidden_layer_sizes=best_arch,
                                 activation=best_activation,
                                 alpha=best_alpha, solver="adam",
                                 max_iter=150, random_state=42))
    ])

    final_pipe.fit(X_tr, y_tr)

    train_auc = roc_auc_score(y_tr,  final_pipe.predict_proba(X_tr)[:, 1])
    val_auc   = roc_auc_score(y_val, final_pipe.predict_proba(X_val)[:, 1])
    gap = train_auc - val_auc

    mlflow.log_metric("train_auc",   round(train_auc, 5))
    mlflow.log_metric("val_auc",     round(val_auc,   5))
    mlflow.log_metric("overfit_gap", round(gap, 5))

    mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="mlp_pipeline",
        registered_model_name="MLP_FraudDetection"
    )

    pkl_path = "/kaggle/working/mlp_final_pipeline.pkl"
    with open(pkl_path, "wb") as f:
        pickle.dump(final_pipe, f)
    mlflow.log_artifact(pkl_path)

    print(f"[Final] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")
    print(f"  -> {'OVERFIT' if gap > 0.05 else 'OK'}")
    print(f"Run ID: {run.info.run_id}")
    print("✅ Registered as MLP_FraudDetection")

NameError: name 'mlflow' is not defined